In [1]:
import sys 
import os
from pathlib import Path
import pandas as pd

#from word_embeddings import *
import torch

In [2]:
from csv import QUOTE_NONE
import sys
import csv

maxInt = sys.maxsize

while True:
    # decrease the maxInt value by factor 10 
    # as long as the OverflowError occurs.

    try:
        csv.field_size_limit(maxInt)
        break
    except OverflowError:
        maxInt = int(maxInt/10)


base_path = Path(os.path.abspath("")).parents[1] / "dataset_creation" / "data"
datasets = {
    "school_shooters": base_path / "school_shooters.csv",
    "manifestos": base_path / "manifestos.csv",
    "stair_twitter_archive": base_path / "stair_twitter_archive.csv",
    "twitter": base_path / "twitter.csv",
    "stream_of_consciousness": base_path / "stream_of_consciousness.csv"
}

schoolshootersinfo_df = pd.read_csv(datasets["school_shooters"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
#manifesto_df = pd.read_csv(datasets["manifestos"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
stair_twitter_archive_df = pd.read_csv(datasets["stair_twitter_archive"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
twitter_df = pd.read_csv(datasets["twitter"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
stream_of_consciousness_df = pd.read_csv(datasets["stream_of_consciousness"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)

In [3]:
# Set label of school shooter or not
# 0 = NOT school shooter
# 1 = school shooter

schoolshootersinfo_df["label"] = 1
#manifesto_df["shooter"] = 1
stair_twitter_archive_df["label"] = 1
twitter_df["label"] = 0
stream_of_consciousness_df["label"] = 0

In [5]:
""" curr = Path(os.path.abspath("")).parents[0] / "utils"
print(curr) """
sys.path.append("..")
from utils.word_embeddings import preprocess_text, get_glove_word_vectors

c:\Users\Jonas\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
shooter_df = pd.concat([schoolshootersinfo_df[:100], stair_twitter_archive_df[:100]], ignore_index=True)
non_shooter_df = pd.concat([twitter_df[:100], stream_of_consciousness_df[:100]], ignore_index=True)
whole_corpus_df = pd.concat([shooter_df, non_shooter_df], ignore_index=True).sample(frac=1)

In [7]:
# Preprocess text into tokens
whole_corpus_df["text"] = whole_corpus_df["text"].map(lambda a: preprocess_text(a))

c:\Users\Jonas\NTNU-Masters\src\experiments\notebooks\..\utils\word_embeddings.py:45: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


In [8]:
whole_corpus_df = whole_corpus_df[whole_corpus_df["text"].map(len) > 0] # In some cases the text is entirely removed by preprocessing. Drop rows where this is the case
whole_corpus_df.head()

sentence_lengths = [len(t) for t in whole_corpus_df["text"]]
max_len = max(sentence_lengths)

In [9]:
# Find max sentence length and send to embeddings method
# This will return all sentences converted to a matrix of glove word embeddings
# Each sentence is padded to the same length as max_len to facilitate use in the neural net

whole_corpus_df["text"] = whole_corpus_df["text"].map(lambda a: get_glove_word_vectors(a, sentence_length=max_len, emb_dim=50))


In [10]:
#print(Path(os.path.abspath("")).parents[1] / "dataset_creation" / "data" / "all_data_glove_emb.pkl")
#whole_corpus_df.to_pickle(Path(os.path.abspath("")).parents[1] / "dataset_creation" / "data" / "all_data_glove_emb.pkl")

In [11]:


from sklearn.model_selection import train_test_split 

x_train, x_test, y_train, y_test = train_test_split(whole_corpus_df["text"], whole_corpus_df["label"], test_size=0.2)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2)

In [12]:
# Check sizes of tensors to shape input to mlp model
""" shapes = [list(t.shape) for t in whole_corpus_df["text"]]
dic_size = [dims[0] for dims in shapes]
print(shapes)


max_dic_size = max(dic_size)
min_dic_size = min(dic_size)
print(min_dic_size)
print(max_dic_size)
for x in x_train:
    if x.shape[0] == min_dic_size:
        print("x_shape: ", x.shape)
        i = 0
        for _ in x:
            print(_)
            i += 1
            print("i: ", i) """

' shapes = [list(t.shape) for t in whole_corpus_df["text"]]\ndic_size = [dims[0] for dims in shapes]\nprint(shapes)\n\n\nmax_dic_size = max(dic_size)\nmin_dic_size = min(dic_size)\nprint(min_dic_size)\nprint(max_dic_size)\nfor x in x_train:\n    if x.shape[0] == min_dic_size:\n        print("x_shape: ", x.shape)\n        i = 0\n        for _ in x:\n            print(_)\n            i += 1\n            print("i: ", i) '

In [13]:
# Creating datasets for use with dataloaders

from torch.utils.data import DataLoader, Dataset

class TextDataset(Dataset):
    def __init__(self, embeddings, labels, train=False):
        self.embeddings = embeddings
        self.labels = labels

    def __len__(self):
        if len(self.labels) == len(self.embeddings):
            return len(self.labels)
        else:
            return -1

    def __getitem__(self, idx):
        #print("embeddings: \n", self.embeddings[idx])
        #print("labels: \n", self.labels[idx])

        return self.embeddings[idx], self.labels[idx]

train_set = TextDataset(x_train.to_numpy(), y_train.to_numpy())
val_set = TextDataset(x_val.to_numpy(), y_val.to_numpy())
test_set = TextDataset(x_test.to_numpy(), y_test.to_numpy())

print(train_set.embeddings[3])

tensor([[ 0.2454, -0.7424,  0.3253,  ...,  0.5651,  0.6310,  0.1988],
        [ 0.1189,  0.1525, -0.0821,  ..., -0.5751, -0.2667,  0.9212],
        [-0.0044, -0.3414, -0.2273,  ..., -0.2526, -0.2948,  0.6110],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]])


In [14]:
train_loader = DataLoader(train_set, batch_size=1, shuffle=True)
val_loader = DataLoader(val_set, batch_size=1, shuffle=False)

""" feats, label = next(iter(train_loader))
feats, label = next(iter(train_loader))

print(feats, label) """

' feats, label = next(iter(train_loader))\nfeats, label = next(iter(train_loader))\n\nprint(feats, label) '

In [18]:
# design model

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"


emb_dim = 50
out_dim = 1

class TextClassifier(nn.Module):
    def __init__(self):
        super(TextClassifier, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=(3,3), padding=(1,1))
        self.relu1 = nn.ReLU()
        
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), padding=(1, 1))
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2))

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=1, kernel_size=(3, 3), padding=(1, 1))
        self.pool3 = nn.MaxPool2d(kernel_size=(2, 2))

        self.flat = nn.Flatten()

        self.fc3 = nn.Linear(2664, 1)
        #self.drop = nn.Dropout(0.5)

        #self.fc4 = nn.Linear(512, 1)
        self.sig = nn.Sigmoid()



        """ self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=(3, 3), padding=(1, 1))
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2))
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=1, kernel_size=(3, 3), padding=(1, 1))
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2)) """
        #self.linear = nn.Linear

    def forward(self, x):
        #print(x.shape)
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        #print("x after maxpool2 (2,2): ", x.shape)
        x = self.conv3(x)
        x = self.pool3(x)
        #print("x after maxpool3 (2,2): ", x.shape)
        x = self.flat(x)
        #print("x after flat: ", x.shape)
        x = self.fc3(x)
        #print("x after linear1: ", x.shape)
        #x = self.drop(x)
        #print("x after drop: ", x.shape)
        #x = self.fc4(x)
        #print("x after linear2: ", x.shape)
        x = self.sig(x)

        #print(x[0])
        return x[0]

model = TextClassifier()
model.to(device)

""" model = nn.Sequential()
model.add(nn.Conv1d(filters=32, kernel_size=8, activation='relu'))
model.add(nn.MaxPool1d(pool_size=2))
model.add(nn.Flatten())
model.add(nn.Dense(10, activation='relu'))
model.add(nn.Dense(1, activation='sigmoid')) """

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [19]:
# Set up training steps and epoch 
from typing import Type


def run_epoch(epoch_i):
    running_loss = 0.
    last_loss = 0.

    for i, data in enumerate(train_loader):
        inputs, labels = data
        labels = labels.to(torch.float32)
        
        inputs = inputs.to(device)
        labels = labels.to(device)
        #print(labels)
        #print(inputs, labels)
        #print(i, inputs, labels)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels.to(torch.float32))
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        # Update reported loss values every 50 steps
        if i % 50 == 49:
            last_loss = running_loss / 50
            #print(f"batch {i+1} loss: {last_loss}")
            #tb_x = epoch_i * len(train_loader) + i + 1
            #tb_writer.add_scalar("Loss/train", last_loss, tb_x)
            running_loss = 0.
        
    return last_loss

In [20]:
# Initializing in a separate cell so we can easily add more epochs to the same run
from datetime import datetime
import wandb
import random

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
#writer = SummaryWriter('runs/mlp_shooters_{}'.format(timestamp))
epoch_number = 0

EPOCHS = 10

best_vloss = 1_000_000.


wandb.init(
    # set the wandb project where this run will be logged
    project="cnn-glove-features-predict-shooters",
    
    # track hyperparameters and run metadata
    config={
    "learning_rate": 0.01,
    "architecture": "CNN",
    "dataset": "school-shooters-vs-non-school-shooters",
    "epochs": 10,
    }
)


for epoch in range(EPOCHS):
    #print('EPOCH {}:'.format(epoch + 1))

    # Make sure gradient tracking is on, and do a pass over the data
    model.train(True)
    avg_loss = run_epoch(epoch)

    # We don't need gradients on to do reporting
    model.train(False)

    running_vloss = 0.0
    for i, vdata in enumerate(val_loader):
        vinputs, vlabels = vdata
        voutputs = model(vinputs)
        vloss = loss_fn(voutputs, vlabels.to(torch.float32))
        running_vloss += vloss

    avg_vloss = running_vloss / (i + 1)
    #print('LOSS train {} valid {}'.format(avg_loss, avg_vloss))
    
    wandb.log({"avg_eloss": avg_loss, "avg_vloss": avg_vloss})

    # Log the running loss averaged per batch
    # for both training and validation

    # Track best performance, and save the model's state
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        model_path = 'model_{}_{}'.format(timestamp, epoch_number)
        torch.save(model.state_dict(), model_path)

wandb.finish()

avg_eloss,▁▂▄▄▄▄▄▄▄█
avg_vloss,▂▁▆▅█▅▅▅▅▇
avg_eloss,0.89937
avg_vloss,0.79016
